# Part 2 — Complementary graph fusion of GRN predictions

This notebook fuses predictions from **two or more GRN inference methods**. It preserves the main logic of the research implementation while removing dataset-specific folder discovery and experiment loops.

The fusion score combines:

1. strongest normalized method score;
2. strongest within-method rank score;
3. cross-method support;
4. local topology of the union network (common neighbors, Jaccard, Adamic–Adar);
5. Node2Vec-style structural embedding similarity.

The reference network is used **only for evaluation**, never to calculate the fusion score.

> **Directionality:** the original research notebook evaluated edges as undirected (`DIRECTED=False`). This option is exposed below. If the benchmark treats TF→target and target→TF as distinct edges, set `DIRECTED=True`. For topology/embeddings, an undirected projection is still used because the structural features implemented here are symmetric.


In [ ]:
from pathlib import Path
from itertools import combinations
import random
import numpy as np
import pandas as pd
import networkx as nx
from gensim.models import Word2Vec
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    f1_score, precision_score, recall_score
)

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
print("Project root:", PROJECT_ROOT)


## Configuration


In [ ]:
# ---------- USER SETTINGS ----------
PREDICTION_FILES = {
    "Method_A": PROJECT_ROOT / "data/example/predictions/Method_A.csv",
    "Method_B": PROJECT_ROOT / "data/example/predictions/Method_B.csv",
}
REFERENCE_FILE = PROJECT_ROOT / "data/example/refNetwork.csv"  # set to None to skip evaluation
OUTPUT_DIR = PROJECT_ROOT / "outputs/part2"

DIRECTED = False
TOP_FRAC_PER_METHOD = 0.10
CANDIDATE_MODE = "method_union_all_predictions"  # or "method_top_union", "full_gene_pairs"

FUSION_WEIGHTS = {
    "max_method_score": 0.10,
    "max_rank_score": 0.30,
    "graph_topology_score": 0.10,
    "support_count_norm": 0.20,
    "node2vec_edge_similarity_norm": 0.30,
}

# Node2Vec-style random-walk settings
EMBEDDING_DIM = 64
WALK_LENGTH = 20
NUM_WALKS = 50
WINDOW = 10
P = 1.0
Q = 1.0
RANDOM_SEED = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Load and standardize prediction files


In [ ]:
def canonical_edge(a, b, directed=False):
    a, b = str(a).strip(), str(b).strip()
    if a == b:
        return None
    if directed:
        return (a, b)
    return (a, b) if a <= b else (b, a)


def normalize_01(values):
    x=np.asarray(values,dtype=float)
    if x.size == 0:
        return x
    lo,hi=np.nanmin(x),np.nanmax(x)
    if hi-lo < 1e-12:
        return np.ones_like(x)
    return (x-lo)/(hi-lo)


def load_prediction(path, directed=False):
    df=pd.read_csv(path)
    lookup={str(c).strip().lower(): c for c in df.columns}
    # Flexible common aliases.
    def first(names):
        for n in names:
            if n in lookup: return lookup[n]
        return None
    c1=first(["gene1","source","source_gene","regulator","tf"])
    c2=first(["gene2","target","target_gene"])
    cw=first(["edgeweight","edge_weight","weight","score","importance","confidence","shap_proba"])
    if c1 is None or c2 is None or cw is None:
        raise ValueError(f"{path}: could not identify gene/score columns. Rename them to Gene1,Gene2,EdgeWeight.")
    out=df[[c1,c2,cw]].copy()
    out.columns=["Gene1","Gene2","EdgeWeight"]
    out["EdgeWeight"]=pd.to_numeric(out["EdgeWeight"],errors="coerce")
    out=out.dropna(subset=["EdgeWeight"])
    edges=[canonical_edge(a,b,directed) for a,b in zip(out.Gene1,out.Gene2)]
    out["_edge"]=edges
    out=out[out["_edge"].notna()].copy()
    out[["Gene1","Gene2"]]=pd.DataFrame(out["_edge"].tolist(),index=out.index)
    out=out.groupby(["Gene1","Gene2"],as_index=False)["EdgeWeight"].max()
    out["NormWeight"]=normalize_01(out["EdgeWeight"])
    # Highest raw score gets rank 1; convert rank to [0,1], higher is better.
    rank=out["EdgeWeight"].rank(method="average",ascending=False).to_numpy(float)
    if len(rank)==1:
        out["NormRank"]=1.0
    else:
        out["NormRank"]=1.0-normalize_01(rank)
    return out

predictions={name:load_prediction(path,DIRECTED) for name,path in PREDICTION_FILES.items()}
if len(predictions)<2:
    raise ValueError("Provide at least two prediction files.")
for name,df in predictions.items():
    print(name, len(df), "predicted edges")


## Build the union graph and graph features


In [ ]:
def top_edges(df, frac):
    k=max(1,int(np.ceil(frac*len(df))))
    return set(map(tuple,df.nlargest(k,"EdgeWeight")[["Gene1","Gene2"]].to_numpy()))

method_top={m:top_edges(df,TOP_FRAC_PER_METHOD) for m,df in predictions.items()}
all_genes=set()
for df in predictions.values():
    all_genes.update(df.Gene1.astype(str)); all_genes.update(df.Gene2.astype(str))

# Structural graph is an undirected projection even when regulatory edges are directed.
G=nx.Graph()
G.add_nodes_from(all_genes)
for m,edges in method_top.items():
    for a,b in edges:
        if G.has_edge(a,b):
            G[a][b]["support"] += 1
        else:
            G.add_edge(a,b,support=1)
print("Union graph:",G.number_of_nodes(),"nodes,",G.number_of_edges(),"edges")


def structural_features(G,a,b):
    if a not in G or b not in G:
        return 0.0,0.0,0.0
    na=set(G.neighbors(a)); nb=set(G.neighbors(b))
    cn=len(na & nb)
    union=na | nb
    jac=(cn/len(union)) if union else 0.0
    aa=0.0
    for z in (na & nb):
        d=G.degree(z)
        if d>1: aa += 1.0/np.log(d)
    return float(cn),float(jac),float(aa)


## Node2Vec-style structural embeddings


In [ ]:
def node2vec_walk(G,start,walk_length,p=1.0,q=1.0,rng=None):
    if rng is None: rng=np.random.default_rng()
    walk=[str(start)]
    while len(walk)<walk_length:
        cur=walk[-1]
        nbrs=list(G.neighbors(cur))
        if not nbrs: break
        if len(walk)==1:
            nxt=nbrs[int(rng.integers(len(nbrs)))]
        else:
            prev=walk[-2]
            weights=[]
            for x in nbrs:
                if x==prev: w=1.0/p
                elif G.has_edge(prev,x): w=1.0
                else: w=1.0/q
                weights.append(w)
            prob=np.asarray(weights,float); prob/=prob.sum()
            nxt=nbrs[int(rng.choice(len(nbrs),p=prob))]
        walk.append(str(nxt))
    return walk


def learn_embeddings(G,dimensions=64,walk_length=20,num_walks=50,window=10,p=1.0,q=1.0,seed=0):
    if G.number_of_edges()==0: return {}
    rng=np.random.default_rng(seed)
    nodes=list(G.nodes())
    walks=[]
    for _ in range(num_walks):
        order=rng.permutation(len(nodes))
        for idx in order:
            walks.append(node2vec_walk(G,nodes[idx],walk_length,p,q,rng))
    model=Word2Vec(
        sentences=walks, vector_size=dimensions, window=window,
        min_count=1, sg=1, workers=1, seed=seed, epochs=5
    )
    return {str(n):model.wv[str(n)] for n in nodes if str(n) in model.wv}

embeddings=learn_embeddings(G,EMBEDDING_DIM,WALK_LENGTH,NUM_WALKS,WINDOW,P,Q,RANDOM_SEED)
print("Embeddings learned for",len(embeddings),"nodes")


## Construct candidate edges and calculate the complementary fusion score


In [ ]:
def all_pairs(genes,directed=False):
    genes=sorted(set(genes))
    if directed:
        return [(a,b) for a in genes for b in genes if a!=b]
    return list(combinations(genes,2))

if CANDIDATE_MODE=="full_gene_pairs":
    candidates=all_pairs(all_genes,DIRECTED)
elif CANDIDATE_MODE=="method_top_union":
    candidates=sorted(set().union(*method_top.values()))
elif CANDIDATE_MODE=="method_union_all_predictions":
    candidates=sorted(set().union(*[set(map(tuple,df[["Gene1","Gene2"]].to_numpy())) for df in predictions.values()]))
else:
    raise ValueError("Unknown CANDIDATE_MODE")

weight_lookup={m:{(r.Gene1,r.Gene2):float(r.NormWeight) for r in df.itertuples()} for m,df in predictions.items()}
rank_lookup={m:{(r.Gene1,r.Gene2):float(r.NormRank) for r in df.itertuples()} for m,df in predictions.items()}

rows=[]
for a,b in candidates:
    ws=np.array([weight_lookup[m].get((a,b),0.0) for m in predictions],float)
    rs=np.array([rank_lookup[m].get((a,b),0.0) for m in predictions],float)
    cn,jac,aa=structural_features(G,a,b)
    if a in embeddings and b in embeddings:
        va,vb=embeddings[a],embeddings[b]
        denom=np.linalg.norm(va)*np.linalg.norm(vb)
        emb_sim=float(np.dot(va,vb)/denom) if denom>0 else 0.0
    else:
        emb_sim=0.0
    row={
        "Gene1":a,"Gene2":b,
        "method_weight_max":float(ws.max(initial=0)),
        "method_rank_max":float(rs.max(initial=0)),
        "method_support_all":int((ws>0).sum()),
        "common_neighbors":cn,"jaccard":jac,"adamic_adar":aa,
        "node2vec_edge_similarity":emb_sim,
    }
    for m,w,r in zip(predictions,ws,rs):
        row[f"{m}_norm_score"]=w; row[f"{m}_rank_score"]=r
    rows.append(row)
feat=pd.DataFrame(rows)

feat["support_count_norm"]=feat["method_support_all"]/len(predictions)
for col in ["common_neighbors","jaccard","adamic_adar","node2vec_edge_similarity"]:
    feat[col+"_norm"]=normalize_01(feat[col])
feat["graph_topology_score"]=feat[["common_neighbors_norm","jaccard_norm","adamic_adar_norm"]].mean(axis=1)
feat["node2vec_edge_similarity_norm"]=normalize_01(feat["node2vec_edge_similarity"])

feat["fusion_score"]=(
    FUSION_WEIGHTS["max_method_score"]*feat["method_weight_max"] +
    FUSION_WEIGHTS["max_rank_score"]*feat["method_rank_max"] +
    FUSION_WEIGHTS["graph_topology_score"]*feat["graph_topology_score"] +
    FUSION_WEIGHTS["support_count_norm"]*feat["support_count_norm"] +
    FUSION_WEIGHTS["node2vec_edge_similarity_norm"]*feat["node2vec_edge_similarity_norm"]
)
# This is a normalized score, NOT a calibrated probability.
feat["normalized_fusion_score"]=normalize_01(feat["fusion_score"])
feat=feat.sort_values("fusion_score",ascending=False).reset_index(drop=True)

fused_file=OUTPUT_DIR/"fused_edges.csv"
feat.to_csv(fused_file,index=False)
print("Saved:",fused_file)
display(feat.head(10))


## Optional evaluation against a reference GRN


In [ ]:
def load_reference(path,directed=False):
    ref=pd.read_csv(path)
    if not {"Gene1","Gene2"}.issubset(ref.columns):
        raise ValueError("Reference file must contain Gene1 and Gene2 columns.")
    out=set()
    for a,b in zip(ref.Gene1,ref.Gene2):
        e=canonical_edge(a,b,directed)
        if e is not None: out.add(e)
    return out


def evaluate_score_table(score_df,score_col,ref_edges,genes,directed=False,top_frac=0.10):
    universe=all_pairs(genes,directed)
    lookup={(r.Gene1,r.Gene2):float(getattr(r,score_col)) for r in score_df.itertuples()}
    y=np.array([int(e in ref_edges) for e in universe],int)
    s=np.array([lookup.get(e,0.0) for e in universe],float)
    auroc=float(roc_auc_score(y,s)) if len(np.unique(y))>1 else np.nan
    aupr=float(average_precision_score(y,s)) if len(np.unique(y))>1 else np.nan
    # Binary metrics use top fraction of all ranked universe edges.
    k=max(1,int(np.ceil(top_frac*len(universe))))
    idx=np.argsort(s)[-k:]
    yp=np.zeros_like(y); yp[idx]=1
    return {
        "AUROC":auroc,"AUPR":aupr,
        "MCC":float(matthews_corrcoef(y,yp)),
        "F1":float(f1_score(y,yp,zero_division=0)),
        "Precision":float(precision_score(y,yp,zero_division=0)),
        "Recall":float(recall_score(y,yp,zero_division=0)),
        "TopFraction":top_frac,
    }

if REFERENCE_FILE is not None:
    ref_edges=load_reference(REFERENCE_FILE,DIRECTED)
    eval_rows=[]
    # Individual methods
    for m,df in predictions.items():
        tmp=df[["Gene1","Gene2","NormWeight"]].rename(columns={"NormWeight":"Score"})
        met=evaluate_score_table(tmp,"Score",ref_edges,all_genes,DIRECTED,TOP_FRAC_PER_METHOD)
        met["Model"]=m; eval_rows.append(met)
    # Conventional rank-average baseline
    rank_cols=[f"{m}_rank_score" for m in predictions]
    rank_avg=feat[["Gene1","Gene2"]].copy()
    rank_avg["Score"]=feat[rank_cols].mean(axis=1)
    met=evaluate_score_table(rank_avg,"Score",ref_edges,all_genes,DIRECTED,TOP_FRAC_PER_METHOD)
    met["Model"]="RankAverage"; eval_rows.append(met)
    # Proposed fusion
    fused_eval=feat[["Gene1","Gene2","normalized_fusion_score"]].rename(columns={"normalized_fusion_score":"Score"})
    met=evaluate_score_table(fused_eval,"Score",ref_edges,all_genes,DIRECTED,TOP_FRAC_PER_METHOD)
    met["Model"]="ComplementaryGraphFusion"; eval_rows.append(met)

    evaluation=pd.DataFrame(eval_rows).set_index("Model")
    evaluation.to_csv(OUTPUT_DIR/"evaluation_summary.csv")
    display(evaluation.sort_values("AUPR",ascending=False))
else:
    print("REFERENCE_FILE=None: evaluation skipped.")


## Files produced

- `outputs/part2/fused_edges.csv` — all candidate edges ranked by the complementary graph-fusion score.
- `outputs/part2/evaluation_summary.csv` — individual methods, rank-average baseline, and graph-fusion metrics when a reference GRN is provided.

### Important interpretation notes

- `normalized_fusion_score` is **not a probability**.
- Raw scores from different GRN methods are normalized within method before fusion.
- If a method produces a statistic where **smaller values mean stronger evidence** (for example, a p-value), transform that score before supplying the CSV.
- The default `DIRECTED=False` reproduces the direction handling in the original research implementation. Set it to `True` for a directional GRN benchmark.
